# Semana 46 · ¿Qué es la mecánica cuántica?

Este cuaderno acompaña a la sesión de proyecto de la semana 46: números complejos, la doble rendija con amplitudes, la llegada de electrones de uno en uno y la partícula en una caja.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Números complejos en Python

Python trae los números complejos de serie. La unidad imaginaria se escribe `1j` (en ingeniería se usa la letra $j$ en lugar de $i$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z1 = 3 + 4j
z2 = 1 - 1j
print(f"z1 + z2 = {z1 + z2}")
print(f"z1 · z2 = {z1 * z2}")
print(f"|z1| = {abs(z1)},  fase de z1 = {np.degrees(np.angle(z1)):.2f}°")
print(f"i · i = {1j * 1j}")
print(f"e^(iπ) + 1 = {np.exp(1j * np.pi) + 1:.2e}   (casi 0: el error es de redondeo)")

In [ ]:
def flecha(ax, z, color, etiqueta, origen=0):
    ax.annotate("", xy=(origen + z).real * np.array([1, 0]) + (origen + z).imag * np.array([0, 1]),
                xytext=(np.real(origen), np.imag(origen)), arrowprops=dict(arrowstyle="->", color=color, lw=2))
    ax.text((origen + z).real, (origen + z).imag, " " + etiqueta, color=color)

fig, axs = plt.subplots(1, 2, figsize=(11, 5))
a, b = 2 + 1j, 0.5 + 1.5j
flecha(axs[0], a, "tab:blue", "a"); flecha(axs[0], b, "tab:orange", "b", origen=a); flecha(axs[0], a + b, "k", "a + b")
axs[0].set_title("Sumar: punta con cola")
u = np.exp(1j * np.radians(30)); v = 1.5 * np.exp(1j * np.radians(60))
flecha(axs[1], u, "tab:blue", "u (1, 30°)"); flecha(axs[1], v, "tab:orange", "v (1,5; 60°)"); flecha(axs[1], u * v, "k", "u·v (1,5; 90°)")
axs[1].set_title("Multiplicar: longitudes multiplicadas, ángulos sumados")
for ax in axs:
    ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
    ax.set_xlim(-1, 3.5); ax.set_ylim(-0.5, 3); ax.set_aspect("equal"); ax.grid(alpha=0.3)
    ax.set_xlabel("parte real"); ax.set_ylabel("parte imaginaria")
plt.tight_layout(); plt.show()

## 2. La doble rendija con amplitudes

Cada rendija contribuye en un punto $y$ de la pantalla con una amplitud compleja $e^{ikr}/r$, donde $r$ es la distancia desde la rendija (como las ondas en el estanque de la semana 27, pero ahora es una amplitud de probabilidad). Sumamos las dos flechas y elevamos al cuadrado el módulo. Para comparar, calculamos también lo que saldría si sumáramos las probabilidades de cada rendija por separado (sin interferencia).

In [ ]:
lam = 50e-12          # longitud de onda de De Broglie de unos electrones rápidos: 50 pm
k = 2 * np.pi / lam
d = 1e-6              # separación entre rendijas: 1 micra
L = 1.0               # distancia a la pantalla: 1 m
y = np.linspace(-200e-6, 200e-6, 4000)

r1 = np.hypot(L, y - d / 2)
r2 = np.hypot(L, y + d / 2)
psi1 = np.exp(1j * k * r1) / r1
psi2 = np.exp(1j * k * r2) / r2
P_cuantica = np.abs(psi1 + psi2) ** 2
P_sin_interferencia = np.abs(psi1) ** 2 + np.abs(psi2) ** 2
envolvente = np.sinc(y / 120e-6) ** 2       # anchura finita de cada rendija (difracción), simplificada
P_cuantica *= envolvente; P_sin_interferencia *= envolvente
P_cuantica /= P_cuantica.sum(); P_sin_interferencia /= P_sin_interferencia.sum()

plt.figure(figsize=(10, 4))
plt.plot(y * 1e6, P_cuantica, label="|ψ₁ + ψ₂|²: con interferencia")
plt.plot(y * 1e6, P_sin_interferencia, label="|ψ₁|² + |ψ₂|²: sin interferencia")
plt.xlabel("posición en la pantalla (µm)"); plt.ylabel("probabilidad"); plt.legend(); plt.grid(alpha=0.3); plt.show()
print(f"separación entre franjas prevista λL/d = {lam * L / d * 1e6:.0f} µm")

## 3. Electrones de uno en uno

Ahora «disparamos» electrones de uno en uno: cada uno llega a un punto **al azar**, elegido con la probabilidad $|\psi_1 + \psi_2|^2$. Para elegir al azar con una probabilidad dada usamos `rng.choice` con pesos. Dibujamos los impactos en una pantalla de dos dimensiones (en vertical, al azar, como si las rendijas fueran largas).

In [ ]:
rng = np.random.default_rng(46)
fig, axs = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, n in zip(axs, [20, 200, 2000, 20000]):
    xs = rng.choice(y, size=n, p=P_cuantica) * 1e6
    ys = rng.uniform(0, 1, n)
    ax.scatter(xs, ys, s=max(0.5, 30 / np.sqrt(n)), c="w")
    ax.set_facecolor("black"); ax.set_title(f"{n} electrones"); ax.set_xlabel("µm"); ax.set_yticks([])
plt.tight_layout(); plt.show()

Con 20 electrones, los puntos parecen al azar. Con miles, aparecen las franjas. Así lo vieron Tonomura y su equipo en 1989, en una pantalla real.

**¿Y si miramos por qué rendija pasa?** Si un detector nos dijera por qué rendija pasa cada electrón, ya no habría dos caminos indistinguibles: cada electrón tendría una sola flecha, y las probabilidades se sumarían sin interferir.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, P, titulo in [(axs[0], P_cuantica, "sin mirar por qué rendija"), (axs[1], P_sin_interferencia, "mirando por qué rendija")]:
    xs = rng.choice(y, size=20000, p=P) * 1e6
    ax.hist(xs, bins=200, color="tab:purple"); ax.set_title(titulo); ax.set_xlabel("µm")
plt.tight_layout(); plt.show()

## 4. Una partícula en una caja

Las ondas estacionarias que caben en una caja de longitud $L$ son $\psi_n(x) = \sqrt{2/L}\,\sin(n\pi x/L)$, con energías $E_n = n^2h^2/(8mL^2)$. Calculamos las de un electrón en una caja de 1 nm (más o menos el tamaño de una molécula grande).

In [ ]:
h, m_e, eV = 6.626e-34, 9.109e-31, 1.602e-19
Lc = 1e-9
x = np.linspace(0, Lc, 500)
fig, ax = plt.subplots(figsize=(8, 6))
for n in range(1, 5):
    E = n**2 * h**2 / (8 * m_e * Lc**2) / eV
    psi = np.sqrt(2 / Lc) * np.sin(n * np.pi * x / Lc)
    escala = 0.25 / np.sqrt(2 / Lc) * 1.5
    ax.axhline(E, color="gray", lw=0.6)
    ax.plot(x * 1e9, E + escala * psi**2 * Lc / 2, label=f"n = {n}: E = {E:.2f} eV")
ax.set_xlabel("posición en la caja (nm)"); ax.set_ylabel("energía (eV); encima, |ψ|²")
ax.legend(fontsize=8); ax.set_title("Electrón en una caja de 1 nm"); plt.show()

Cada nivel tiene una forma de onda distinta, y $|\psi|^2$ (lo que se dibuja encima de cada nivel) dice dónde es probable encontrar el electrón. En el nivel $n = 2$, la probabilidad de encontrarlo justo en el centro de la caja es **cero**, aunque puede estar a los dos lados: algo imposible para una partícula clásica que va y viene. Las diferencias de energía, de unos pocos eV, corresponden a fotones de luz visible o ultravioleta (semana 30): por eso muchas moléculas tienen color.

## 5. Experimenta

1. En la doble rendija, cambia la longitud de onda a 100 pm. ¿Qué les pasa a las franjas? Con $\lambda = h/(mv)$, ¿significa eso electrones más rápidos o más lentos?
2. Pon a una de las rendijas una amplitud la mitad que la otra (multiplica `psi2` por 0,5). ¿Desaparecen los ceros de las franjas?
3. ¿Qué longitud de onda tiene el fotón que emite el electrón de la caja al pasar de $n = 2$ a $n = 1$? ¿Y de $n = 3$ a $n = 2$?

In [ ]:
# Tu experimento aquí